# Segundo Entregable — Análisis Comparativo de Datasets y Selección para el Modelo de Fuga de Clientes

**Proyecto:** Predicción de Fuga de Clientes en Marketplaces  
**Dataset base:** Brazilian E-Commerce Public Dataset — Olist  
**Entregable:** W14, semana del 5 de mayo  

---

Este notebook complementa el primer entregable. Su objetivo es documentar el análisis comparativo de tres datasets candidatos, justificar la elección del dataset seleccionado, integrarlo al pipeline existente de Olist, y avanzar en las etapas de preprocesamiento, ingeniería de características con texto, y modelado supervisado, siguiendo la planificación establecida en la Sección 6 del entregable anterior.

El hilo conductor se mantiene: predecir si un cliente se fugará (no regresará), con la mejora de incorporar señales de texto estructuradas a partir de reseñas reales.


---

## Sección A — Análisis Comparativo de Datasets Candidatos

Antes de seleccionar un dataset complementario, se evaluaron tres opciones en función de cinco criterios alineados al problema de negocio:

| Criterio | Descripción |
|---|---|
| Afinidad temática | Que el dataset sea de e-commerce o comportamiento de compra |
| Riqueza textual | Presencia de reseñas u otro texto que permita extraer variables de satisfacción |
| Calidad y limpieza | Ausencia de datos anonimizados, vacíos masivos o ruido estructural |
| Compatibilidad técnica | Que sus variables se puedan unir o complementar con Olist sin esfuerzo excesivo |
| Complejidad ML viable | Que permita construir features útiles para clasificación sin requerir infraestructura especializada |

---

### Dataset 1 — Amazon Review Dataset (Kaggle: sahityasahu/amazon-review-dataset)

**Descripción general**  
Colección de reseñas de productos de Amazon. Contiene texto de review, puntuación numérica (1-5 estrellas), identificador de producto, y metadatos del reviewer. Las reseñas son en inglés y corresponden a múltiples categorías de producto.

**Estructura relevante**

| Columna | Tipo | Relevancia para churn |
|---|---|---|
| `reviewText` | Texto libre | Alta — señal directa de satisfacción o problema |
| `overall` | Numérico (1-5) | Alta — equivale al review_score de Olist |
| `verified` | Booleano | Media — indica autenticidad de la compra |
| `asin` | Categórico | Media — ID de producto para cruzar con categoría |
| `summary` | Texto corto | Media — resumen de la reseña |
| `reviewTime` | Fecha | Alta — permite construir recencia |

**Fortalezas**
- Texto disponible en la gran mayoría de registros (tasa de llenado > 80 %).  
- El campo `overall` es equivalente al `review_score` de Olist — permite transferencia directa de la lógica ya construida.  
- Volumen suficiente para aplicar técnicas de NLP sin riesgo de subajuste.  
- La columna `verified` añade una señal de calidad del review no presente en Olist.

**Debilidades**
- No tiene identificador de cliente persistente; solo existe reviewer hash — imposible construir historial RFM real sin heurísticas.  
- Idioma inglés: requeriría traducción o modelos multilingues si se quiere alinear con las reseñas en portugués de Olist.  
- Las categorías de producto son de Amazon, no equivalentes a las de Olist — el cruce semántico requeriría un mapeo manual.  
- No contiene datos de pago, entrega o logística — variables centrales del modelo de churn del proyecto.

**Puntuación: 5.5 / 10**  
El dataset tiene el mejor contenido textual de los tres candidatos, pero su falta de identificador de cliente persistente y su desconexión estructural con las dimensiones de Olist (pagos, entregas) lo limitan para complementar directamente el pipeline.

---

### Dataset 2 — American Express Default Prediction (Kaggle: amex-default-prediction)

**Descripción general**  
Dataset de una competición de Kaggle para predecir la probabilidad de impago de tarjetas de crédito. Contiene series de tiempo de perfiles de clientes con variables financieras anónimas. El archivo `train_labels.csv` proporciona el target binario (default = 1).

**Estructura relevante**

| Prefijo de variable | Cantidad aproximada | Tipo |
|---|---|---|
| `D_*` (Delinquency) | ~30 columnas | Comportamiento de mora |
| `S_*` (Spending) | ~20 columnas | Gasto y consumo |
| `P_*` (Payment) | ~5 columnas | Información de pago |
| `B_*` (Balance) | ~20 columnas | Saldo e información de deuda |
| `R_*` (Risk) | ~10 columnas | Variables de riesgo crediticio |

El train_data.csv pesa aproximadamente 16 GB. El train_labels.csv tiene ~460.000 registros con el label binario.

**Fortalezas**
- Variable objetivo binaria clara (default / no default) — conceptualmente análogo al churn.  
- Alta densidad de features numéricas para ML supervisado clásico.  
- Dataset ampliamente documentado por la comunidad Kaggle con soluciones de referencia en XGBoost, LightGBM y redes neuronales.

**Debilidades**
- Todas las variables están anonimizadas — no hay interpretación de negocio posible: no se sabe qué representa `D_44` ni `B_17`.  
- No contiene texto: ninguna reseña, comentario ni descripción — elimina la posibilidad de NLP.  
- El dominio es financiero/crédito, no e-commerce — la distancia semántica con Olist es máxima.  
- El tamaño del dataset (16 GB) es prohibitivo en entornos de Colab gratuito sin manejo de datos por chunks.  
- Su integración con el pipeline de Olist requeriría fabricar un "puente" artificial entre dominios incompatibles.

**Puntuación: 2.5 / 10**  
Dataset valioso como problema de ML puro, pero completamente desalineado con el proyecto. No aporta variables de e-commerce, no tiene texto y su anonimización impide cualquier narrativa de negocio coherente con Olist.

---

### Dataset 3 — H&M Personalized Fashion Recommendations (Kaggle: h-and-m-personalized-fashion-recommendations)

**Descripción general**  
Dataset de la competición H&M de Kaggle para sistemas de recomendación. Incluye tres archivos principales: `articles.csv` con metadatos de productos (tipo, color, grupo de prenda, descripción textual), `customers.csv` con atributos demográficos y de membresía, y `transactions_train.csv` con el historial completo de compras entre septiembre de 2018 y septiembre de 2020.

**Estructura relevante**

| Archivo | Columnas clave | Aporte al proyecto |
|---|---|---|
| `articles.csv` (106k productos) | `article_id`, `product_type_name`, `colour_group_name`, `section_name`, `detail_desc` | Descripción textual del producto |
| `customers.csv` (1.37M clientes) | `customer_id`, `age`, `club_member_status`, `fashion_news_frequency` | Perfil de cliente para RFM ampliado |
| `transactions_train.csv` | `t_dat` (fecha), `customer_id`, `article_id`, `price`, `sales_channel_id` | Historial completo para recencia, frecuencia, valor |

**Fortalezas**
- Tiene identificador de cliente persistente y real — permite construir RFM genuino, igual que Olist.  
- La columna `detail_desc` en `articles.csv` provee texto descriptivo por producto: se puede agrupar por cliente las descripciones de lo que compra, generando un perfil textual.  
- Las variables de cliente (`club_member_status`, `fashion_news_frequency`) son señales directas de engagement — análogas a indicadores de churn.  
- La estructura de tres archivos (clientes, productos, transacciones) es casi idéntica a la arquitectura de Olist — la curva de aprendizaje del merge ya está resuelta.  
- El volumen (1.37M clientes, 2 años de transacciones) es robusto sin ser prohibitivo.  
- El dominio es e-commerce de retail — compatible con la narrativa del proyecto.

**Debilidades**
- No tiene columna de reseñas explícita con texto de opinión del cliente: el texto disponible es la descripción del producto, no la voz del cliente.  
- La ausencia de review_score impide replicar directamente la señal de satisfacción de Olist — requiere construir un proxy a partir del comportamiento de recompra.  
- El dominio es moda (fashion), con dinámicas de temporalidad y estacionalidad más pronunciadas que en un marketplace generalista.

**Puntuación: 7.5 / 10**  
Es el dataset más estructuralmente compatible con Olist. Permite ampliar el modelo con features de comportamiento longitudinal real, variables demográficas de cliente y texto descriptivo de producto. Su arquitectura de tres archivos replica el esquema relacional ya manejado en el primer entregable.

---

### Decisión de Selección

El Dataset 3 — H&M es el seleccionado para la integración en el segundo entregable.

La razón principal no es únicamente la puntuación, sino la coherencia arquitectónica: el pipeline de merge y agregación RFM construido en las Secciones 4 del primer entregable se puede aplicar casi directamente. Adicionalmente, la columna `fashion_news_frequency` junto a `club_member_status` actúan como señales de engagement que permiten enriquecer la definición de churn más allá del umbral temporal de 180 días.

El texto disponible en `detail_desc` se usará para construir un perfil de categoría semántica por cliente, lo que enriquece la feature `most_frequent_category` existente con información descriptiva, no solo con el código de categoría.


---

## Sección B — Carga del Dataset H&M

El dataset se descarga desde la competición de Kaggle. Para usarlo en Google Colab el procedimiento recomendado es descargar manualmente el ZIP desde la página de la competición y subirlo al entorno de Colab, o usar la API de Kaggle si se tiene configurada la autenticación.

### Instrucciones de descarga manual

1. Ingresar a: https://www.kaggle.com/competitions/h-and-m-personalized-fashion-recommendations/data  
2. Descargar los archivos: `articles.csv`, `customers.csv`, `transactions_train.csv`  
3. Subir los tres archivos al entorno de Colab mediante la celda de carga o montando Google Drive.

Las celdas siguientes asumen que los tres archivos están disponibles en la ruta `/content/` dentro del entorno de Colab, al igual que los archivos de Olist.


In [1]:
# Limpiar todo lo descargado
!rm -rf /content/hm_data
!rm -f h-and-m-personalized-fashion-recommendations.zip
!df -h /content  # Ver espacio disponible después

Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   21G   87G  20% /


In [2]:
!du -sh /content/*
!df -h

55M	/content/sample_data
Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   21G   87G  20% /
tmpfs            64M     0   64M   0% /dev
shm             5.7G     0  5.7G   0% /dev/shm
/dev/root       2.0G  1.2G  748M  63% /usr/sbin/docker-init
tmpfs           6.4G  328K  6.4G   1% /var/colab
/dev/sda1       114G  109G  5.3G  96% /kaggle/input
tmpfs           6.4G     0  6.4G   0% /proc/acpi
tmpfs           6.4G     0  6.4G   0% /proc/scsi
tmpfs           6.4G     0  6.4G   0% /sys/firmware


In [6]:
import os

# Token de acceso directo (formato KGAT_)
KAGGLE_API_TOKEN = "KGAT_44142ea6c947fb913769610d87108e83"

os.makedirs('/root/.kaggle', exist_ok=True)

# Para tokens KGAT_ se guarda directo en access_token, NO en kaggle.json
with open('/root/.kaggle/access_token', 'w') as f:
    f.write(KAGGLE_API_TOKEN)

os.chmod('/root/.kaggle/access_token', 0o600)

# Instalar versión actualizada de kaggle que soporte tokens KGAT_
!pip install kaggle --upgrade -q

# Verificar autenticación
!kaggle competitions list 2>&1 | head -5

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.4/95.4 kB 6.0 MB/s eta 0:00:00
ref                                                                              deadline             category         reward  teamCount  userHasEntered  
-------------------------------------------------------------------------------  -------------------  --------  -------------  ---------  --------------  
https://www.kaggle.com/competitions/passenger-screening-algorithm-challenge      2017-12-15 23:59:00  Featured  1,500,000 Usd        518           False  
https://www.kaggle.com/competitions/zillow-prize-1                               2018-01-10 15:59:00  Featured  1,200,000 Usd       3770           False  
https://www.kaggle.com/competitions/data-science-bowl-2017                       2017-04-12 23:59:00  Featured  1,000,000 Usd       1972           False  


In [7]:
import os, json

os.makedirs('/root/.kaggle', exist_ok=True)

# Pega aquí los valores del kaggle.json descargado
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({"username": "tu_usuario", "key": "tu_key_nueva"}, f)

os.chmod('/root/.kaggle/kaggle.json', 0o600)
!pip install kaggle --upgrade -q
!kaggle competitions list 2>&1 | head -5

ref                                                                              deadline             category         reward  teamCount  userHasEntered  
-------------------------------------------------------------------------------  -------------------  --------  -------------  ---------  --------------  
https://www.kaggle.com/competitions/passenger-screening-algorithm-challenge      2017-12-15 23:59:00  Featured  1,500,000 Usd        518           False  
https://www.kaggle.com/competitions/zillow-prize-1                               2018-01-10 15:59:00  Featured  1,200,000 Usd       3770           False  
https://www.kaggle.com/competitions/data-science-bowl-2017                       2017-04-12 23:59:00  Featured  1,000,000 Usd       1972           False  


In [8]:
# Descargar solo los archivos necesarios, uno por uno
!kaggle competitions download \
    -c h-and-m-personalized-fashion-recommendations \
    -f articles.csv

!kaggle competitions download \
    -c h-and-m-personalized-fashion-recommendations \
    -f customers.csv

!kaggle competitions download \
    -c h-and-m-personalized-fashion-recommendations \
    -f transactions_train.csv

# Descomprimir
!unzip -q articles.csv.zip -d /content/
!unzip -q customers.csv.zip -d /content/
!unzip -q transactions_train.csv.zip -d /content/
!rm *.zip

!ls -lh /content/*.csv

100% 4.26M/4.26M [00:00<00:00, 192MB/s]

100% 97.9M/97.9M [00:00<00:00, 111MB/s]

100% 584M/584M [00:05<00:00, 108MB/s]

-rw-r--r-- 1 root root  35M Jan 17  2022 /content/articles.csv
-rw-r--r-- 1 root root 198M Jan 17  2022 /content/customers.csv
-rw-r--r-- 1 root root 3.3G Jan 17  2022 /content/transactions_train.csv


In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Ruta base — archivos descargados directamente en /content/
path_hm = '/content/'

df_hm_articulos = pd.read_csv(f'{path_hm}articles.csv', dtype={'article_id': str})
df_hm_clientes = pd.read_csv(f'{path_hm}customers.csv')
df_hm_transacciones = pd.read_csv(f'{path_hm}transactions_train.csv', dtype={'article_id': str})

df_hm_transacciones['t_dat'] = pd.to_datetime(df_hm_transacciones['t_dat'])

print('Dimensiones de los DataFrames de H&M:')
print(f'  Articulos:      {df_hm_articulos.shape}')
print(f'  Clientes:       {df_hm_clientes.shape}')
print(f'  Transacciones:  {df_hm_transacciones.shape}')
print(f'  Rango temporal: {df_hm_transacciones["t_dat"].min().date()} a {df_hm_transacciones["t_dat"].max().date()}')


Dimensiones de los DataFrames de H&M:
  Articulos:      (105542, 25)
  Clientes:       (1371980, 7)
  Transacciones:  (31788324, 5)
  Rango temporal: 2018-09-20 a 2020-09-22


---

## Sección C — Exploración Inicial y Calidad del Dataset H&M

Se aplica la misma metodología de auditoría del primer entregable: revisión de tipos, nulos, duplicados y estadísticas descriptivas para cada archivo. Esto garantiza consistencia metodológica en el proyecto.


In [ ]:
# Exploración de articles.csv
print('=' * 55)
print('ANALISIS: articles.csv')
print('=' * 55)
print(df_hm_articulos.info())
print()
print('Valores nulos por columna:')
nulos_art = df_hm_articulos.isnull().sum()
print(nulos_art[nulos_art > 0])
print()
print('Duplicados:', df_hm_articulos.duplicated().sum())
print()
display(df_hm_articulos[['article_id', 'product_type_name', 'colour_group_name',
                           'section_name', 'detail_desc']].head())


In [ ]:
# Exploración de customers.csv
print('=' * 55)
print('ANALISIS: customers.csv')
print('=' * 55)
print(df_hm_clientes.info())
print()
print('Valores nulos por columna:')
nulos_cli = df_hm_clientes.isnull().sum()
print(nulos_cli[nulos_cli > 0])
print()
print('Duplicados:', df_hm_clientes.duplicated().sum())
print()
# Distribución de variables categóricas clave
print('--- Distribución de club_member_status ---')
print(df_hm_clientes['club_member_status'].value_counts())
print()
print('--- Distribución de fashion_news_frequency ---')
print(df_hm_clientes['fashion_news_frequency'].value_counts())


In [ ]:
# Exploración de transactions_train.csv
print('=' * 55)
print('ANALISIS: transactions_train.csv')
print('=' * 55)
print(df_hm_transacciones.info())
print()
print('Valores nulos por columna:')
nulos_tx = df_hm_transacciones.isnull().sum()
print(nulos_tx[nulos_tx > 0] if nulos_tx.sum() > 0 else 'No hay valores nulos.')
print()
print('Duplicados:', df_hm_transacciones.duplicated().sum())
print()
print('Estadísticas descriptivas de precio:')
display(df_hm_transacciones[['price']].describe())
print()
print('Distribución de canal de venta (sales_channel_id):')
print(df_hm_transacciones['sales_channel_id'].value_counts())


---

## Sección D — Análisis Exploratorio Visual (EDA) — H&M

Se replican las cuatro visualizaciones del primer entregable sobre el nuevo dataset: evolución temporal, distribución de precios, geografía de clientes (aquí reemplazada por distribución de edad) y canal de venta. Esto permite contrastar el comportamiento de ambos datasets.


In [ ]:
sns.set_theme(style='whitegrid', palette='muted')

# 1. Evolución mensual de transacciones
df_hm_transacciones['mes'] = df_hm_transacciones['t_dat'].dt.to_period('M')
transacciones_mes = df_hm_transacciones['mes'].value_counts().sort_index()

plt.figure(figsize=(14, 5))
transacciones_mes.plot(kind='line', marker='o', color='royalblue', linewidth=2)
plt.title('Evolucion Temporal de Transacciones H&M (Mensual)', fontsize=14)
plt.xlabel('Mes')
plt.ylabel('Cantidad de Transacciones')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# 2. Distribución de edad de clientes H&M
plt.figure(figsize=(12, 5))
sns.histplot(df_hm_clientes['age'].dropna(), bins=50, kde=True, color='teal', edgecolor='white', linewidth=0.3)
plt.title('Distribucion de Edad de Clientes H&M', fontsize=14)
plt.xlabel('Edad')
plt.ylabel('Frecuencia')
plt.tight_layout()
plt.show()

print(f'Edad promedio: {df_hm_clientes["age"].mean():.1f} años')
print(f'Edad mediana:  {df_hm_clientes["age"].median():.1f} años')
print(f'Clientes sin edad registrada: {df_hm_clientes["age"].isnull().sum():,}')


In [ ]:
# 3. Top 10 tipos de producto más vendidos
top_tipos = (df_hm_transacciones
             .merge(df_hm_articulos[['article_id', 'product_type_name']], on='article_id', how='left')
             ['product_type_name'].value_counts().head(10))

plt.figure(figsize=(12, 5))
sns.barplot(x=top_tipos.values, y=top_tipos.index,
            hue=top_tipos.index, palette='magma', legend=False)
plt.title('Top 10 Tipos de Producto mas Transaccionados', fontsize=14)
plt.xlabel('Cantidad de Transacciones')
plt.ylabel('Tipo de Producto')
plt.tight_layout()
plt.show()


In [ ]:
# 4. Distribución del precio en escala logarítmica (equivalente a payment_value en Olist)
plt.figure(figsize=(12, 5))
sns.histplot(df_hm_transacciones['price'], bins=80, kde=False,
             color='steelblue', edgecolor='white', linewidth=0.3)
plt.yscale('log')
plt.title('Distribucion del Precio por Transaccion H&M (Escala Logaritmica)', fontsize=14)
plt.xlabel('Precio (normalizado)')
plt.ylabel('Frecuencia (Log)')
plt.tight_layout()
plt.show()


---

## Sección E — Ingeniería de Características — Dataset H&M

Se construye el DataFrame de modelado para H&M siguiendo la misma lógica RFM del primer entregable, con dos adiciones importantes:

1. **Variables de engagement del cliente:** `club_member_status` y `fashion_news_frequency` son señales directas de relación del cliente con la plataforma. Un cliente inactivo en el club o que dejó de recibir noticias de moda puede ser un indicador de fuga.

2. **Perfil de canal:** el campo `sales_channel_id` permite distinguir si el cliente compra exclusivamente en tienda física (canal 1) o en línea (canal 2), lo que puede correlacionarse con el riesgo de abandono en el canal digital.

La variable objetivo `churn` se define con el mismo criterio del primer entregable: clientes con recencia superior a 180 días desde la última transacción.


In [ ]:
# 1. Fecha de referencia (última fecha del dataset)
fecha_ref_hm = df_hm_transacciones['t_dat'].max()
print(f'Fecha de referencia (max del dataset): {fecha_ref_hm.date()}')

# 2. Agregación de transacciones por cliente — métricas RFM
df_hm_rfm = df_hm_transacciones.groupby('customer_id').agg(
    recency=('t_dat', lambda x: (fecha_ref_hm - x.max()).days),
    frequency=('t_dat', 'count'),
    monetary=('price', 'sum'),
    canal_predominante=('sales_channel_id', lambda x: x.mode()[0])
).reset_index()

print(f'Clientes con al menos una transaccion: {len(df_hm_rfm):,}')
print()
print('Estadísticas de metricas RFM:')
display(df_hm_rfm[['recency', 'frequency', 'monetary']].describe())


In [ ]:
# 3. Unir con información demográfica del cliente
df_hm_features = pd.merge(df_hm_rfm, df_hm_clientes, on='customer_id', how='left')

# 4. Codificación de variables de engagement
# club_member_status: ACTIVE, PRE-CREATE, LEFT CLUB -> ordinal de engagement
club_map = {'ACTIVE': 2, 'PRE-CREATE': 1, 'LEFT CLUB': 0}
df_hm_features['club_status_cod'] = df_hm_features['club_member_status'].map(club_map).fillna(1)

# fashion_news_frequency: REGULARLY, MONTHLY, None -> ordinal de engagement
news_map = {'Regularly': 2, 'Monthly': 1, 'NONE': 0, 'None': 0}
df_hm_features['news_freq_cod'] = df_hm_features['fashion_news_frequency'].map(news_map).fillna(0)

# 5. Imputar edad con la mediana (variable continua con ~20% nulos)
mediana_edad = df_hm_features['age'].median()
df_hm_features['age'] = df_hm_features['age'].fillna(mediana_edad)

# 6. Variable objetivo
df_hm_features['churn'] = (df_hm_features['recency'] > 180).astype(int)

print(f'Dimensiones del DataFrame de modelado H&M: {df_hm_features.shape}')
print()
print('Distribucion de la variable Churn:')
churn_dist = df_hm_features['churn'].value_counts(normalize=True) * 100
print(churn_dist.rename({0: 'No Churn (0)', 1: 'Churn (1)'}))
print()
display(df_hm_features[['customer_id', 'recency', 'frequency', 'monetary',
                          'age', 'club_status_cod', 'news_freq_cod',
                          'canal_predominante', 'churn']].head())


---

## Sección F — Construcción de Perfil Textual por Cliente

La columna `detail_desc` en `articles.csv` contiene la descripción textual de cada producto. A partir de ella se construye un perfil de categoría semántica por cliente: para cada cliente se concatenan las descripciones de todos los productos que compró, lo que permite extraer las temáticas predominantes de su comportamiento de compra.

Este perfil textual se usa para dos propósitos:

1. **Feature de categoría enriquecida:** reemplaza la feature `most_frequent_category` del primer entregable por una representación más densa que captura múltiples categorías.  
2. **Preparación para NLP:** el texto concatenado puede ser tokenizado y vectorizado (TF-IDF o embeddings) para entrenar modelos que detecten patrones semánticos asociados al churn.

Esta sección implementa la versión TF-IDF, que es coherente con el nivel de la asignatura en esta etapa del semestre.


In [ ]:
# 1. Limpiar y preparar las descripciones de artículos
df_hm_articulos['detail_desc'] = df_hm_articulos['detail_desc'].fillna('sin descripcion')
df_hm_articulos['product_type_name'] = df_hm_articulos['product_type_name'].fillna('desconocido')

# Unir transacciones con descripción del artículo
df_tx_desc = df_hm_transacciones.merge(
    df_hm_articulos[['article_id', 'detail_desc', 'product_type_name']],
    on='article_id',
    how='left'
)

# 2. Construir el texto acumulado por cliente
# Se usa product_type_name (más corto y limpio) para el perfil semántico
perfil_texto = (df_tx_desc.groupby('customer_id')['product_type_name']
                .agg(lambda x: ' '.join(x.dropna().astype(str).str.lower()))
                .reset_index()
                .rename(columns={'product_type_name': 'perfil_texto_producto'}))

print(f'Clientes con perfil textual generado: {len(perfil_texto):,}')
print()
print('Ejemplo de perfil textual para los primeros 3 clientes:')
display(perfil_texto.head(3))


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 3. Vectorización TF-IDF del perfil textual
# max_features=50 para mantener el modelo interpretable en esta etapa
tfidf = TfidfVectorizer(max_features=50, min_df=5, stop_words='english')

X_texto = tfidf.fit_transform(perfil_texto['perfil_texto_producto'])

print(f'Dimensiones de la matriz TF-IDF: {X_texto.shape}')
print()
print('Terminos con mayor peso global en los perfiles de cliente:')
suma_tfidf = np.asarray(X_texto.sum(axis=0)).flatten()
vocabulario = tfidf.get_feature_names_out()
top_terminos = pd.Series(suma_tfidf, index=vocabulario).sort_values(ascending=False).head(15)
print(top_terminos)


In [ ]:
# 4. Unir las features TF-IDF al DataFrame de modelado
df_tfidf = pd.DataFrame(X_texto.toarray(),
                         columns=[f'tfidf_{t}' for t in vocabulario],
                         index=perfil_texto['customer_id'])

df_hm_features_nlp = df_hm_features.set_index('customer_id').join(df_tfidf, how='left').reset_index()

# Imputar nulos en columnas TF-IDF (clientes sin texto disponible)
cols_tfidf = [c for c in df_hm_features_nlp.columns if c.startswith('tfidf_')]
df_hm_features_nlp[cols_tfidf] = df_hm_features_nlp[cols_tfidf].fillna(0)

print(f'Dimensiones finales del DataFrame con features NLP: {df_hm_features_nlp.shape}')
print('Primeras 5 columnas TF-IDF añadidas:')
print(cols_tfidf[:5])


---

## Sección G — Preprocesamiento y Encoding

Esta sección ejecuta la pipeline de preprocesamiento planificada para la semana W07 del primer entregable. Se aplica sobre el dataset H&M, pero la misma lógica es trasladable directamente al dataset Olist.

Los pasos son:
1. Selección de features numéricas y categóricas relevantes.  
2. Escalado estándar (`StandardScaler`) de variables continuas.  
3. Codificación ordinal de variables categóricas ya mapeadas.  
4. Split estratificado train/test 80-20, respetando el desbalance de clases.


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# 1. Selección de features para el modelo
# Se excluyen identificadores, fechas intermedias y columnas de texto sin codificar
features_modelo = ['recency', 'frequency', 'monetary', 'age',
                   'club_status_cod', 'news_freq_cod', 'canal_predominante'] + cols_tfidf

# Asegurar que no haya nulos en las features seleccionadas
df_modelo = df_hm_features_nlp[features_modelo + ['churn']].dropna()
print(f'Registros disponibles para modelado (sin nulos): {len(df_modelo):,}')
print(f'Features totales: {len(features_modelo)}')

# 2. Separación de X e y
X = df_modelo[features_modelo].values
y = df_modelo['churn'].values

print(f'\nDistribucion de clases en el dataset de modelado:')
print(f'  No Churn (0): {(y == 0).sum():,} ({(y==0).mean()*100:.1f}%)')
print(f'  Churn (1):    {(y == 1).sum():,} ({(y==1).mean()*100:.1f}%)')


In [ ]:
# 3. Split estratificado 80-20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Escalado con StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Dimensiones del conjunto de entrenamiento:')
print(f'  X_train: {X_train_scaled.shape}  |  y_train: {y_train.shape}')
print(f'  X_test:  {X_test_scaled.shape}   |  y_test:  {y_test.shape}')


---

## Sección H — Clasificador Base: Regresion Logistica (Neurona Simple)

Siguiendo la planificación de la semana W07, se implementa el clasificador base usando Regresion Logistica. Este modelo actúa como la "neurona simple" del curso: es un clasificador lineal que aprende una frontera de decisión como combinación ponderada de las features de entrada.

**Por qué la regresion logistica como baseline:**
- Es computacionalmente ligera y reproducible.  
- Sus coeficientes son directamente interpretables — cada peso indica la contribución de cada feature a la probabilidad de churn.  
- Establece un piso de desempeño contra el cual se compararán modelos más complejos.  
- El parámetro `class_weight='balanced'` compensará el desbalance de clases identificado en la Seccion E.

La metrica principal sera el F1-score sobre la clase positiva (churn=1), ya que el accuracy en datasets desbalanceados puede ser engañoso.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, ConfusionMatrixDisplay)

# Entrenamiento del clasificador base
modelo_lr = LogisticRegression(
    class_weight='balanced',
    max_iter=500,
    random_state=42,
    C=1.0
)
modelo_lr.fit(X_train_scaled, y_train)

# Predicciones
y_pred_lr = modelo_lr.predict(X_test_scaled)
y_prob_lr = modelo_lr.predict_proba(X_test_scaled)[:, 1]

print('--- Resultados: Regresion Logistica (Baseline) ---')
print()
print(classification_report(y_test, y_pred_lr, target_names=['No Churn', 'Churn']))
print(f'AUC-ROC: {roc_auc_score(y_test, y_prob_lr):.4f}')


In [ ]:
# Visualización de la matriz de confusión
fig, ax = plt.subplots(figsize=(6, 5))
cm_lr = confusion_matrix(y_test, y_pred_lr)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_lr, display_labels=['No Churn', 'Churn'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Matriz de Confusion — Regresion Logistica', fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# Interpretación de coeficientes del modelo lineal
coefs = pd.Series(modelo_lr.coef_[0], index=features_modelo).sort_values(key=abs, ascending=False)

plt.figure(figsize=(12, 6))
coefs.head(20).sort_values().plot(kind='barh', color='steelblue', edgecolor='white')
plt.title('Top 20 Features por Magnitud del Coeficiente — Regresion Logistica', fontsize=13)
plt.xlabel('Coeficiente (magnitud = importancia)')
plt.axvline(0, color='black', linewidth=0.8, linestyle='--')
plt.tight_layout()
plt.show()

print('Las 5 features con mayor peso positivo (aumentan probabilidad de churn):')
print(coefs[coefs > 0].head(5))
print()
print('Las 5 features con mayor peso negativo (reducen probabilidad de churn):')
print(coefs[coefs < 0].head(5))


---

## Sección I — Red Neuronal Densa (MLP) con Keras

Siguiendo el contenido de la semana W08 (introduccion a redes neuronales profundas), se implementa un Perceptron Multicapa (MLP) sobre el mismo dataset de modelado. La arquitectura replica la estructura presentada en el material `12_std_Notes_Intro_DL.ipynb`:

- **Capa de entrada:** una neurona por cada feature (dimensionalidad = número de features del modelo).  
- **Capas ocultas:** dos capas densas con activacion ReLU — introducen no-linealidad para capturar relaciones entre features que la regresion logistica no puede modelar.  
- **Capa de salida:** una neurona con activacion Sigmoide, que produce la probabilidad de churn (entre 0 y 1).

El entrenamiento usa `binary_crossentropy` como función de pérdida (equivalente al caso binario de `sparse_categorical_crossentropy` del material del curso), y el optimizador Adam, que es una versión adaptativa del descenso de gradiente estocástico visto en clase.


In [ ]:
import tensorflow as tf
from tensorflow import keras
print(f'TensorFlow version: {tf.__version__}')

# Semilla para reproducibilidad
tf.random.set_seed(42)
np.random.seed(42)

n_features = X_train_scaled.shape[1]

# Arquitectura del MLP
modelo_mlp = keras.Sequential([
    keras.layers.Input(shape=(n_features,)),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(1, activation='sigmoid')
])

modelo_mlp.summary()


In [ ]:
# Calcular pesos de clase para compensar el desbalance
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
peso_neg = 1.0
peso_pos = n_neg / n_pos
class_weight_dict = {0: peso_neg, 1: peso_pos}
print(f'Peso clase No Churn (0): {peso_neg:.2f}')
print(f'Peso clase Churn (1):    {peso_pos:.2f}')

# Compilación
modelo_mlp.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc')]
)

# Entrenamiento
historia = modelo_mlp.fit(
    X_train_scaled, y_train,
    epochs=30,
    batch_size=512,
    validation_split=0.15,
    class_weight=class_weight_dict,
    verbose=1
)


In [ ]:
# Curvas de aprendizaje
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(historia.history['loss'], label='Entrenamiento', color='royalblue')
axes[0].plot(historia.history['val_loss'], label='Validacion', color='tomato')
axes[0].set_title('Curva de Perdida (Binary Crossentropy)', fontsize=13)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].plot(historia.history['auc'], label='Entrenamiento', color='royalblue')
axes[1].plot(historia.history['val_auc'], label='Validacion', color='tomato')
axes[1].set_title('Curva AUC-ROC durante Entrenamiento', fontsize=13)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('AUC')
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import classification_report, roc_auc_score

# Evaluación sobre el conjunto de prueba
y_prob_mlp = modelo_mlp.predict(X_test_scaled).flatten()
y_pred_mlp = (y_prob_mlp >= 0.5).astype(int)

print('--- Resultados: Red Neuronal MLP (Keras) ---')
print()
print(classification_report(y_test, y_pred_mlp, target_names=['No Churn', 'Churn']))
print(f'AUC-ROC: {roc_auc_score(y_test, y_prob_mlp):.4f}')


---

## Sección J — Comparativa de Modelos e Interpretación de Resultados

Se consolidan las métricas de los modelos entrenados en una tabla comparativa. La comparativa permite:

- Identificar si la complejidad adicional del MLP aporta ganancia real sobre el baseline lineal.
- Seleccionar el modelo a llevar al siguiente entregable para ajuste de hiperparámetros.
- Fundamentar la decisión con métricas apropiadas para datos desbalanceados (F1, AUC-ROC).


In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

# Función de métricas reutilizable
def tabla_metricas(nombre, y_real, y_pred, y_prob):
    return {
        'Modelo': nombre,
        'Precision (Churn)': round(precision_score(y_real, y_pred, pos_label=1), 4),
        'Recall (Churn)': round(recall_score(y_real, y_pred, pos_label=1), 4),
        'F1-score (Churn)': round(f1_score(y_real, y_pred, pos_label=1), 4),
        'AUC-ROC': round(roc_auc_score(y_real, y_prob), 4)
    }

resultados = [
    tabla_metricas('Regresion Logistica (Baseline)', y_test, y_pred_lr, y_prob_lr),
    tabla_metricas('MLP Keras (2 capas ocultas)',    y_test, y_pred_mlp, y_prob_mlp),
]

df_comparativa = pd.DataFrame(resultados).set_index('Modelo')
display(df_comparativa)


In [ ]:
# Curvas ROC comparativas
from sklearn.metrics import roc_curve

fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
fpr_mlp, tpr_mlp, _ = roc_curve(y_test, y_prob_mlp)

plt.figure(figsize=(9, 6))
plt.plot(fpr_lr, tpr_lr,
         label=f'Regresion Logistica (AUC = {roc_auc_score(y_test, y_prob_lr):.3f})',
         color='royalblue', linewidth=2)
plt.plot(fpr_mlp, tpr_mlp,
         label=f'MLP Keras (AUC = {roc_auc_score(y_test, y_prob_mlp):.3f})',
         color='tomato', linewidth=2)
plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Clasificador aleatorio')
plt.title('Curvas ROC — Comparativa de Modelos', fontsize=13)
plt.xlabel('Tasa de Falsos Positivos')
plt.ylabel('Tasa de Verdaderos Positivos')
plt.legend(loc='lower right')
plt.grid(alpha=0.4)
plt.tight_layout()
plt.show()


---

## Sección K — Desafios Identificados y Planificacion hacia el Entregable Final

### Desafios del Dataset H&M

**1. Ausencia de review_score explícito**  
A diferencia de Olist, H&M no tiene una puntuación numérica de satisfacción del cliente. El proxy construido (club_status_cod, news_freq_cod) captura el engagement, pero no la satisfacción post-compra. En el siguiente entregable se evaluará si la frecuencia de compra de artículos similares puede actuar como señal implícita de satisfacción.

**2. Texto descriptivo vs. texto evaluativo**  
La columna `detail_desc` contiene texto del producto, no del cliente. Esto limita la aplicación de técnicas de análisis de sentimiento. Las features TF-IDF construidas capturan preferencias de categoría, no insatisfacción expresada directamente. Esta limitación se documentará en las conclusiones.

**3. Desbalance de clases**  
Al igual que en Olist (~70/30), el dataset H&M presentará desbalance dependiendo del umbral temporal elegido. La estrategia `class_weight='balanced'` aplicada en ambos modelos mitiga este problema pero no lo elimina.

**4. Clientes sin historial previo al corte temporal**  
Clientes que empezaron a comprar tarde en el período cubierto tendrán alta recencia por construcción, no por insatisfacción. Se documentará esta limitación como herencia del diseño del primer entregable.

---

### Planificacion — Semanas Restantes

| Semana | Tarea |
|---|---|
| W09 (mar 31 - abr 2) | Entrenar Random Forest y Gradient Boosting sobre el dataset H&M. Comparar las cuatro curvas ROC en una sola figura. Generar la tabla de importancia de variables para el mejor modelo de ensemble |
| W10 (abr 7-9) | Implementar un regresor para predecir la frecuencia futura de compra del cliente. Evaluar con MAE y R². Usar el score predicho como feature adicional del clasificador de churn |
| W11 (abr 14-16) | Análisis de importancia de variables final. Seleccionar el modelo definitivo para el entregable. Redactar conclusiones de negocio: qué tipo de cliente tiene mayor riesgo y qué acción de marketing corresponde |
| W12 (abr 21-23) | Consolidacion del notebook. Revisar que todo corre de inicio a fin sin errores. Completar celdas de texto e interpretacion |
| W13 (abr 28-30) | Revision integral. Preparar el avance 2 con el pipeline completo documentado |

---

### Resumen de Avance del Proyecto

| Componente | Estado |
|---|---|
| Seleccion y justificacion del dataset complementario | Completado — Dataset H&M |
| Carga y exploración inicial H&M | Completado |
| EDA visual H&M | Completado |
| Ingeniería de características RFM extendida | Completado |
| Perfil textual por cliente (TF-IDF sobre tipos de producto) | Completado |
| Preprocesamiento y encoding | Completado |
| Clasificador base — Regresion Logistica | Completado |
| Red neuronal MLP con Keras | Completado |
| Comparativa de modelos con curvas ROC | Completado |
| Modelos de ensemble (Random Forest, Gradient Boosting) | Pendiente — W09 |
| Analisis de regresion sobre frecuencia de compra | Pendiente — W10 |
| Interpretacion de importancia de variables | Pendiente — W11 |
| Ajuste de hiperparametros del modelo final | Pendiente — W11 |
| Reduccion dimensional (PCA) y agrupamiento | Pendiente — W15 |
